# MODELIZACIÓN PD: CHALLENGERS DE BOOSTING (XGBOOST Y LIGHTGBM)

Este notebook busca un modelo de **probabilidad de impago (PD)** que ordene mejor a los clientes que el modelo actual
del notebook 05: una regresión logística Ridge con splines (Gini externo 0,4225, Brier 0,1435).

**Candidatos:** XGBoost y LightGBM, dos implementaciones de *gradient boosting* de árboles.

**Reglas del experimento:**

* Mismo tablón, misma partición train/test 70/30 y mismos 5 folds que el notebook 05, para que la comparación sea 1 a 1.
* Métrica principal: **ROC-AUC**, reportada también como **Gini** (`2·AUC − 1`). Controles: **Brier**, **log loss** y
  **calibración por deciles**, porque la PD alimenta la pérdida esperada y su nivel tiene que ser creíble.
* El ganador se elige **solo con validación cruzada** sobre train. El test interno y la validación externa se usan una
  única vez, al final.

## IMPORTAR PAQUETES

**Qué hace:** importa las librerías, fija la semilla y comprueba las versiones de XGBoost y LightGBM.
**Por qué:** las versiones quedan registradas en la salida; un modelo de boosting entrenado con una versión no siempre
se carga igual con otra.

In [1]:
import time
import warnings
from pathlib import Path

import joblib
import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer
from xgboost import XGBClassifier

pd.set_option('display.float_format', lambda v: '%.4f' % v)
warnings.filterwarnings('ignore', message='X does not have valid feature names')

SEMILLA = 42

print(f'scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}')

scikit-learn 1.9.0 | xgboost 3.4.1 | lightgbm 4.7.0


## IMPORTAR LOS DATOS

**Qué hace:** carga el tablón **sin transformar** (15 variables crudas + `target_pd`), el mismo del notebook 05.
**Por qué:** el preprocesado se aplica dentro del pipeline, ajustado solo con los datos de entrenamiento de cada fold.
**Qué esperar:** 83.250 préstamos y una tasa de impago cercana al 20%.

In [2]:
df = pd.read_pickle('../02_datos/03_Entrenamiento/df_tablon_pd_sin_transformar.pkl')
x = df.drop(columns='target_pd')
y = df.target_pd
print(f'{df.shape[0]} préstamos, {x.shape[1]} variables, tasa de impago {y.mean():.1%}')

83250 préstamos, 15 variables, tasa de impago 19.9%


## PARTICIÓN Y VALIDACIÓN CRUZADA

**Qué hace:** separa el 30% como test interno y define los 5 folds estratificados.
**Por qué:** con la misma semilla que el notebook 05, train, test y cada fold contienen exactamente los mismos
préstamos. Así se puede comparar fold a fold contra el modelo con splines.
**Qué esperar:** 58.275 filas de train y 24.975 de test, las dos con tasa de impago de 0,199.

In [3]:
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.3, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

pd.DataFrame({'filas': [len(train_y), len(test_y)], 'tasa_impago': [train_y.mean(), test_y.mean()]},
             index=['train', 'test'])

,filas,tasa_impago
train,58275,0.1990
test,24975,0.1990


## PREPROCESADOR Y MÉTRICAS

**Qué hace:** carga el preprocesador de la fase de transformación (one-hot, ordinal, Yeo-Johnson, escalado) y define
las métricas que se calculan en cada fold.
**Por qué:** `clone` copia la configuración pero no lo aprendido, así que el preprocesador se reajusta en cada fold.
A los árboles no les afectan Yeo-Johnson ni el escalado, porque esas transformaciones no cambian el orden de los valores
y un árbol solo mira el orden para elegir sus cortes. Usar el mismo preprocesador mantiene la comparación pareja.

**Métricas** (todas en la misma pasada de validación cruzada):

| Clave | Qué mide | Mejor si |
|---|---|---|
| `auc` | Ordenamiento: probabilidad de que un moroso tenga más PD que un pagador. Gini = 2·AUC − 1 | mayor |
| `brier` | Error cuadrático medio entre la PD y el desenlace (0/1). Mezcla orden y nivel | menor |
| `log_loss` | Como el Brier, pero castiga mucho más las PD muy seguras y equivocadas | menor |
| `pr_auc` | Área bajo la curva precisión-recall. Complementaria; con 20% de morosos el AUC ya es informativo | mayor |
| `accuracy_05`, `precision_05`, `recall_05`, `f1_05` | Métricas de clasificación **cortando en PD = 0,5** | mayor |

Las métricas `_05` dependen del corte. En una PD casi nadie supera 0,5, así que se reportan solo como referencia.
sklearn devuelve el Brier y el log loss con signo negativo (`neg_...`) para que "mayor sea mejor" en todas. La función
`resumen_cv` les vuelve a dar el signo positivo.

In [4]:
preprocesador = clone(joblib.load('../05_modelos/preprocesador.joblib'))

METRICAS = {
    'auc': 'roc_auc',
    'brier': 'neg_brier_score',
    'log_loss': 'neg_log_loss',
    'pr_auc': 'average_precision',
    'accuracy_05': 'accuracy',
    'precision_05': 'precision',
    'recall_05': 'recall',
    'f1_05': 'f1',
}
METRICAS_NEGATIVAS = {'brier', 'log_loss'}


def resumen_cv(resultados, prefijo='test_'):
    """Media, desvío y media de train de cada métrica de cross_validate, con el signo corregido."""
    filas = {}
    for metrica in METRICAS:
        signo = -1 if metrica in METRICAS_NEGATIVAS else 1
        cv_vals = signo * np.asarray(resultados[f'{prefijo}{metrica}'])
        train_vals = signo * np.asarray(resultados[f'train_{metrica}'])
        filas[metrica] = {'media_cv': cv_vals.mean(), 'std_cv': cv_vals.std(), 'media_train': train_vals.mean()}
    tabla = pd.DataFrame(filas).T
    tabla['gap_train_cv'] = np.where(tabla.index.isin(list(METRICAS_NEGATIVAS)),
                                     tabla['media_cv'] - tabla['media_train'],
                                     tabla['media_train'] - tabla['media_cv'])
    return tabla


def ks_estadistico(y_true, y_pred):
    """Máxima distancia entre las distribuciones acumuladas de PD de morosos y pagadores."""
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return np.max(tpr - fpr)

## BASELINE: LOGÍSTICA CON SPLINES (MODELO ACTUAL)

**Qué hace:** reconstruye el pipeline ganador del notebook 05 (splines cúbicas con 8 nudos en `principal`, `dti`,
`imp_cuota` y `tipo_interes`, más logística Ridge con `C ≈ 0,316`) y lo evalúa con los 5 folds y todas las métricas.
**Por qué:** es la vara que hay que superar. No se vuelve a buscar hiperparámetros, se usan los ya elegidos.
**Qué esperar:** un AUC medio de alrededor de 0,710 (0,7098 en el notebook 05) y el AUC de cada fold, que sirve
después para la comparación pareada fold a fold. `gap_train_cv` positivo indica que train rinde mejor que CV, señal
de sobreajuste si es grande.

In [5]:
COLUMNAS_SPLINE = ['yj__principal', 'yj__dti', 'yj__imp_cuota', 'ss__tipo_interes']

pipe_splines = Pipeline([
    ('preprocesador', clone(preprocesador)),
    ('splines', ColumnTransformer(
        [('spl', SplineTransformer(n_knots=8, degree=3, knots='quantile', include_bias=False), COLUMNAS_SPLINE)],
        remainder='passthrough',
        verbose_feature_names_out=False,
    ).set_output(transform='pandas')),
    ('algoritmo', LogisticRegression(solver='saga', l1_ratio=0, C=10 ** -0.5, max_iter=3000,
                                     random_state=SEMILLA)),
])

inicio = time.time()
cv_splines = cross_validate(pipe_splines, train_x, train_y, cv=cv, scoring=METRICAS,
                            return_train_score=True, n_jobs=-1)
print(f'Validación cruzada del baseline en {time.time() - inicio:.0f} s')

auc_folds = pd.DataFrame({'Logística + splines': cv_splines['test_auc']},
                         index=[f'fold {i}' for i in range(1, 6)])
display(auc_folds.T)
resumen_cv(cv_splines)

Validación cruzada del baseline en 25 s


,fold 1,fold 2,fold 3,fold 4,fold 5
Logística + splines,0.7023,0.7066,0.7131,0.7177,0.7094


,media_cv,std_cv,media_train,gap_train_cv
auc,0.7098,0.0053,0.7123,0.0025
brier,0.1449,0.0007,0.1445,0.0004
log_loss,0.4536,0.0026,0.4523,0.0013
pr_auc,0.3714,0.0068,0.3751,0.0037
accuracy_05,0.8034,0.0012,0.8037,0.0002
precision_05,0.5541,0.0266,0.5604,0.0062
recall_05,0.0621,0.0051,0.0624,0.0003
f1_05,0.1116,0.0087,0.1122,0.0006


**Lectura.** El baseline reproduce exactamente los AUC por fold del notebook 05 (de 0,7023 a 0,7177; media 0,7098,
desvío 0,0053). La partición y los folds son idénticos, así que la comparación posterior es pareada. El sobreajuste
es mínimo: train supera a CV en solo 0,0025 de AUC.

Las métricas cortando en 0,5 muestran por qué no sirven para una PD. La *accuracy* de 0,803 es prácticamente la de
predecir "no impaga" para todos (0,801), y el *recall* de 0,062 indica que solo el 6% de los morosos supera una PD de 0,5.
El modelo no es malo: ordena bien (AUC 0,71). El corte en 0,5 no tiene sentido con un 20% de morosos.

**La vara para los challengers:** AUC CV > 0,7098 (Gini 0,4196), sin empeorar el Brier (0,1449) ni el log loss (0,4536).